In [1]:
import pandas as pd

In [3]:
df_gmv = pd.read_csv("./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv", encoding='utf-8-sig')


### Thống kê sdt của gmv và lead

In [5]:
# Hàm đếm độ dài số điện thoại
def phone_length_stats(df, phone_col):
    # Chuyển về string
    phones = (
        df[phone_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Chỉ giữ lại số
    phones = phones.str.replace(r"\D", "", regex=True)

    # Tính độ dài
    lengths = phones.str.len()

    # Đếm tần suất
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("Phone Length")
        .reset_index(name="Count")
    )

    return stats

# GMV
print("===== GMV =====")
print(phone_length_stats(df_gmv, "Phone"))

# Leads
print("\n===== LEADS =====")
print(phone_length_stats(df_leads, "Số điện thoại"))

===== GMV =====
   Phone Length  Count
0            11    345
1            12    101
2            13     17

===== LEADS =====
   Phone Length  Count
0             0    229
1             4      1
2            10      9
3            11  14372
4            12   1560
5            13    160
6            14      3
7            15      2


In [8]:
def phone_length_examples(df, phone_col, n=5):
    temp = pd.DataFrame({
        "Raw": df[phone_col].fillna("").astype(str)
    })

    # Chỉ dùng để tính độ dài
    temp["Clean"] = (
        temp["Raw"]
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    temp["Length"] = temp["Clean"].str.len()

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")

        samples = (
            temp[temp["Length"] == length]
            [["Raw", "Clean"]]
            .drop_duplicates()
            .head(n)
        )

        print(samples.to_string(index=False))
        
print("===== GMV =====")
phone_length_examples(df_gmv, "Phone")

print("\n===== LEADS =====")
phone_length_examples(df_leads, "Số điện thoại")

===== GMV =====

===== Length = 11 =====
         Raw       Clean
84-908224672 84908224672
84-909517679 84909517679
84-964678701 84964678701
84-389970625 84389970625
84-938593961 84938593961

===== Length = 12 =====
          Raw        Clean
420-734602066 420734602066
81-7031882708 817031882708
81-8035704573 818035704573
81-7085394685 817085394685
81-8051986529 818051986529

===== Length = 13 =====
           Raw         Clean
49-15750418390 4915750418390
49-17644461422 4917644461422
49-15205315239 4915205315239
49-17641893888 4917641893888
49-17687465203 4917687465203

===== LEADS =====

===== Length = 0 =====
    Raw Clean
             
QR ZALO      
      -      
  MÃ QR      
QR Zalo      

===== Length = 4 =====
  Raw Clean
16:33  1633

===== Length = 10 =====
        Raw      Clean
82-66831308 8266831308
84-35449883 8435449883
84-77705527 8477705527
82-66248699 8266248699
84-56562448 8456562448

===== Length = 11 =====
         Raw       Clean
84-919249237 84919249237
84-9654685

### Thống kê uid của gmv và leads

In [6]:
def uid_length_stats(df, uid_col):
    # Chuẩn hóa UID
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()      # hoặc .str.lower()
    )

    # Tính độ dài
    lengths = uids.str.len()

    # Thống kê
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("UID Length")
        .reset_index(name="Count")
    )

    return stats


# GMV
print("===== GMV UID =====")
print(uid_length_stats(df_gmv, "UID"))

# Leads
print("\n===== LEADS UID =====")
print(uid_length_stats(df_leads, "UID"))

===== GMV UID =====
   UID Length  Count
0          10    463

===== LEADS UID =====
   UID Length  Count
0           0    761
1           1      1
2           2      1
3           9      4
4          10   8759
5          11      5
6          12   6805


In [7]:
def uid_length_examples(df, uid_col, n=5):
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    temp = pd.DataFrame({
        "UID": uids,
        "Length": uids.str.len()
    })

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")
        print(
            temp[temp["Length"] == length]["UID"]
            .drop_duplicates()
            .head(n)
            .tolist()
        )

print("GMV")
uid_length_examples(df_gmv, "UID")

print("\nLEADS")
uid_length_examples(df_leads, "UID")

GMV

===== Length = 10 =====
['3311001921', '3179205818', '3311304274', '3310902627', '3309271098']

LEADS

===== Length = 0 =====
['']

===== Length = 1 =====
['`']

===== Length = 2 =====
['L8']

===== Length = 9 =====
['309845189', '310072108', '307486256', '307272816']

===== Length = 10 =====
['3309599159', '3309599173', '3309599181', '3309622104', '3288358317']

===== Length = 11 =====
['310991875.0', '305383787.0', '158072687.0', '305580541.0', '305825571.0']

===== Length = 12 =====
['3310947243.0', '3310952712.0', '3293618952.0', '3310947249.0', '3310952706.0']


### Clean sdt và phone của cả 2


In [9]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!
